In [28]:
import numpy as np
import pandas as pd
import os
from datetime import datetime, date
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.vector_ar.vecm import VECM
from scipy.linalg import eig
print(f"modules have been imported.")

modules have been imported.


In [11]:
# Step 1: to simulate dummy macroeconomic data for 4 economies (T=100 periods)

np.random.seed(42)

num_periods = 100
economies = ['SG','US','CN','EU']
variables = ['gdp', 'inf', 'int']

columns = [f"{eco}_{var}" for eco in economies for var in variables]
data = pd.DataFrame(np.random.randn(num_periods, len(columns)), columns = columns)
display(data)

# 2. define fixed trade weights matrix (rows sum to 1, diags to 0)
# Rows: SG, US, CN, EU | Cols: SG, US, CN, EU

W = np.array([
    [0.0, 0.3, 0.4, 0.3], # SG trade partners
    [0.1, 0.0, 0.5, 0.4], # US trade partners
    [0.2, 0.4, 0.0, 0.4], # CN trade partners
    [0.2, 0.4, 0.4, 0.0]  # EU trade partners
])

W

,SG_gdp,SG_inf,SG_int,US_gdp,US_inf,US_int,CN_gdp,CN_inf,CN_int,EU_gdp,EU_inf,EU_int
0,0.496714,-0.138264,0.647689,1.523030,-0.234153,-0.234137,1.579213,0.767435,-0.469474,0.542560,-0.463418,-0.465730
1,0.241962,-1.913280,-1.724918,-0.562288,-1.012831,0.314247,-0.908024,-1.412304,1.465649,-0.225776,0.067528,-1.424748
2,-0.544383,0.110923,-1.150994,0.375698,-0.600639,-0.291694,-0.601707,1.852278,-0.013497,-1.057711,0.822545,-1.220844
3,0.208864,-1.959670,-1.328186,0.196861,0.738467,0.171368,-0.115648,-0.301104,-1.478522,-0.719844,-0.460639,1.057122
4,0.343618,-1.763040,0.324084,-0.385082,-0.676922,0.611676,1.031000,0.931280,-0.839218,-0.309212,0.331263,0.975545
...,...,...,...,...,...,...,...,...,...,...,...,...
95,-0.297564,1.375707,-0.150056,0.125576,-0.173072,0.015579,-1.096275,-1.440051,1.594505,-0.846961,-0.991392,-2.153390
96,-0.638962,-1.323090,1.642015,1.009817,-0.688150,2.252436,0.981765,-0.324831,-2.499406,2.290943,-1.389572,-1.645399
97,1.022570,2.439752,1.384273,0.563909,0.594754,0.853416,0.758929,0.281191,0.104201,-0.062593,-0.753965,-0.280675
98,-1.692957,-0.098340,-0.988591,-1.103589,0.179894,1.392002,0.918317,-1.570501,-0.989628,0.940771,-0.982487,-0.224633


array([[0. , 0.3, 0.4, 0.3],
       [0.1, 0. , 0.5, 0.4],
       [0.2, 0.4, 0. , 0.4],
       [0.2, 0.4, 0.4, 0. ]])

In [19]:
# 3. Construct Foreign Variables (X*) for Singapore (SG) as an example
sg_partners_data = data[[col for col in data.columns if not col.startswith('SG')]]
sg_partners_data

# reshape partners data to compute weighted average easily.
sg_w = W[0,1:]  # weights for US, CN, EU
sg_w

gdp_star = data['US_gdp'] * sg_w[0] + data['CN_gdp'] * sg_w[1] + data['EU_gdp'] * sg_w[2]
inf_star = data['US_inf'] * sg_w[0] + data['CN_inf'] * sg_w[1] + data['EU_inf'] * sg_w[2]
int_star = data['US_int'] * sg_w[0] + data['CN_int'] * sg_w[1] + data['EU_int'] * sg_w[2]

X_sg_domestic = data[['SG_gdp', 'SG_inf', 'SG_int']]
X_sg_foreign = pd.DataFrame({'gdp_star' : gdp_star,
                            'inf_star' : inf_star,
                            'int_star': int_star})

print(f"X_sg_domestic is: {display(X_sg_domestic)}")
print(f"X_sg_foreign is: {display(X_sg_foreign)}")


,SG_gdp,SG_inf,SG_int
0,0.496714,-0.138264,0.647689
1,0.241962,-1.913280,-1.724918
2,-0.544383,0.110923,-1.150994
3,0.208864,-1.959670,-1.328186
4,0.343618,-1.763040,0.324084
...,...,...,...
95,-0.297564,1.375707,-0.150056
96,-0.638962,-1.323090,1.642015
97,1.022570,2.439752,1.384273
98,-1.692957,-0.098340,-0.988591


X_sg_domestic is: None


,gdp_star,inf_star,int_star
0,1.251362,0.097703,-0.397750
1,-0.599629,-0.848512,0.253109
2,-0.445287,0.807483,-0.459160
3,-0.203154,-0.037093,-0.222862
4,0.204111,0.268814,0.140479
...,...,...,...
95,-0.654926,-0.925360,-0.003541
96,1.382934,-0.753249,-0.817651
97,0.453966,0.064713,0.213503
98,0.318481,-0.868978,-0.045641


X_sg_foreign is: None


In [25]:
# 4. Fit VARX(1) for Singapore
# Note: Treat contemporaneous foreign variables and their lags as exog
exog_data = pd.concat([X_sg_foreign, X_sg_foreign.shift(1)], axis = 1).dropna()
exog_data.columns = ['gdp_star', 'inf_star', 'int_star', 'gdp_star_lag1', 'inf_star_lag1', 'int_star_lag1']
endog_data = X_sg_domestic.iloc[1:] # Align time index due to lag shift.

print(f"X_sg_foreign is: {display(X_sg_foreign)}")
print(f"X_sg_foreign.shift(1) is: {display(X_sg_foreign.shift(1).dropna())}")
print(f"exog_data is: {display(exog_data)}")

varx_model = VAR(endog_data, exog=exog_data)
varx_results = varx_model.fit(maxlags = 1)
print(varx_results.summary())

,gdp_star,inf_star,int_star
0,1.251362,0.097703,-0.397750
1,-0.599629,-0.848512,0.253109
2,-0.445287,0.807483,-0.459160
3,-0.203154,-0.037093,-0.222862
4,0.204111,0.268814,0.140479
...,...,...,...
95,-0.654926,-0.925360,-0.003541
96,1.382934,-0.753249,-0.817651
97,0.453966,0.064713,0.213503
98,0.318481,-0.868978,-0.045641


X_sg_foreign is: None


,gdp_star,inf_star,int_star
1,1.251362,0.097703,-0.397750
2,-0.599629,-0.848512,0.253109
3,-0.445287,0.807483,-0.459160
4,-0.203154,-0.037093,-0.222862
5,0.204111,0.268814,0.140479
...,...,...,...
95,-0.339369,1.048968,0.822200
96,-0.654926,-0.925360,-0.003541
97,1.382934,-0.753249,-0.817651
98,0.453966,0.064713,0.213503


X_sg_foreign.shift(1) is: None


,gdp_star,inf_star,int_star,gdp_star_lag1,inf_star_lag1,int_star_lag1
1,-0.599629,-0.848512,0.253109,1.251362,0.097703,-0.397750
2,-0.445287,0.807483,-0.459160,-0.599629,-0.848512,0.253109
3,-0.203154,-0.037093,-0.222862,-0.445287,0.807483,-0.459160
4,0.204111,0.268814,0.140479,-0.203154,-0.037093,-0.222862
5,-0.581202,0.753590,1.012937,0.204111,0.268814,0.140479
...,...,...,...,...,...,...
95,-0.654926,-0.925360,-0.003541,-0.339369,1.048968,0.822200
96,1.382934,-0.753249,-0.817651,-0.654926,-0.925360,-0.003541
97,0.453966,0.064713,0.213503,1.382934,-0.753249,-0.817651
98,0.318481,-0.868978,-0.045641,0.453966,0.064713,0.213503


exog_data is: None
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Sat, 18, Jul, 2026
Time:                     16:14:26
--------------------------------------------------------------------
No. of Equations:         3.00000    BIC:                   0.748487
Nobs:                     98.0000    HQIC:                  0.277242
Log likelihood:          -385.069    FPE:                   0.960126
AIC:                   -0.0428295    Det(Omega_mle):        0.717356
--------------------------------------------------------------------
Results for equation SG_gdp
                   coefficient       std. error           t-stat            prob
--------------------------------------------------------------------------------
const                 0.081888         0.093012            0.880           0.379
gdp_star             -0.198692         0.166244           -1.195           0.232
inf_star              0.059982         0

In [36]:
# Create Johansens test

# 1. Simulate a system where 2 variables are explicitly cointegrated (share a trend)
np.random.seed(42)
T = 200

# Common non-stationary stochastic trend (Random Walk)
common_trend = np.cumsum(np.random.normal(0,1,T))
common_trend

# Generate 3 long-term variables (z_t)
z1 = common_trend + np.random.normal(0,0.5, T) # Driven by trend
z2 = 0.8 * common_trend + np.random.normal(0, 0.5, T) # Cointegrated with z1
z3 = np.cumsum(np.random.normal(0, 1, T)) # Independent random walk

Z = np.vstack([z1, z2, z3]).T # shape: (T, 3)
print(f"Z matrix is: {Z}")
print(f"Z shape is: {Z.shape}")

# 2. prepare the VECM Data Arrays
# Z_t (current levels), Z_lag (lagged levels), dZ (differences)
dZ = np.diff(Z, axis = 0)  # \Delta z_t
Z_lag = Z[:-1, :]  # z_{t-1}
print(f"dZ matrix is: {dZ}")
print(f"dZ shape is: {dZ.shape}")
print(f"Z_lag matrix is: {Z_lag}")
print(f"Z_lag shape is: {Z_lag.shape}")

T_effective = dZ.shape[0] # no of rows (199)
k = Z.shape[1] # no. of variables (3).

Z matrix is: [[ 6.75607833e-01 -3.99842507e-01  7.56988617e-01]
 [ 6.38842115e-01 -1.29276300e-02 -1.65176708e-01]
 [ 1.54766401e+00  8.07532562e-01  7.04429213e-01]
 [ 3.05606927e+00  2.04682489e+00  2.06006707e+00]
 [ 1.60618019e+00  1.61097916e+00  2.47350197e+00]
 [ 1.59196539e+00  1.96012730e+00  4.35029779e+00]
 [ 3.89760836e+00  2.37826237e+00  3.57650859e+00]
 [ 4.66441843e+00  3.45483062e+00  2.33185388e+00]
 [ 4.19557492e+00  3.21058867e+00  5.53133636e-01]
 [ 6.40697686e+00  3.84170831e+00  2.04917795e+00]
 [ 4.30263868e+00  3.56956218e+00  2.70354360e+00]
 [ 4.11924649e+00  2.27884989e+00  2.64795893e+00]
 [ 4.27042682e+00  2.26768367e+00  2.92792756e+00]
 [ 2.20584132e+00  2.14295497e+00  1.80243851e+00]
 [-2.40675732e-03  2.90339298e-01  4.24819049e+00]
 [-2.75750540e-02 -6.99891000e-01  4.37741167e+00]
 [-1.80630339e+00 -3.60336640e-01  4.48680647e+00]
 [-1.22405276e+00 -8.26677444e-01  5.21257309e+00]
 [-2.25634930e+00 -1.02128543e+00  5.69358232e+00]
 [-3.38503416e+00 

In [38]:
# 3. Compute Moment Matrices (Cross-products)
# M_00 = Variance of short-run changes
# M_11 = Variance of long-run lagged levels
# M_01 / M_10: Covariance between short-run changes and long run levels.
M_00 = (dZ.T @ dZ) / T_effective
M_11 = (Z_lag.T @ Z_lag) / T_effective
M_01 = (dZ.T @ Z_lag) / T_effective
M_10 = M_01.T

print(f"M_00, M_11, M_01 and M_10 are as follows:")
print(M_00, M_11, M_01, M_10)
print(M_00.shape)

M_00, M_11, M_01 and M_10 are as follows:
[[ 1.41290285  0.67311506 -0.00655059]
 [ 0.67311506  0.97267709  0.05131293]
 [-0.00655059  0.05131293  1.03673309]] [[79.10558135 63.62950556 -5.5509043 ]
 [63.62950556 51.60085275 -4.41781241]
 [-5.5509043  -4.41781241 15.74730922]] [[-0.56493366 -0.31271112 -0.42362823]
 [-0.11917812 -0.38591492 -0.43547854]
 [ 0.35970307  0.32860012 -0.51172585]] [[-0.56493366 -0.11917812  0.35970307]
 [-0.31271112 -0.38591492  0.32860012]
 [-0.42362823 -0.43547854 -0.51172585]]
(3, 3)


In [54]:
# 4. Solve Johansen's Generalized Eigenvalue Problem
# We solve: | \lambda * M_11 - M_10 * (M_00^-1) * M_01 | = 0
M_00_inv = np.linalg.inv(M_00)
A = M_10 @ M_00_inv @ M_01
B = M_11
print(f"M_00_inv is: {M_00_inv}")
print(f"A is: {A}")
print(f"B is: {B}")

# Compute generalized eigenvalues and eigenvectors 
eigenvalues, eigenvectors = eig(A, B)
print(f"eigenvalues, eigenvectors are:\n {eigenvalues},\n {eigenvectors}")

# Real components only, then sort descending by eigenvalue strength
idx = np.argsort(eigenvalues.real)[::-1]
eigenvalues = eigenvalues.real[idx]
eigenvectors = eigenvectors.real[:,idx] # to rearrange the eigenvectors based on the sorting of the eigenvalues computed earlier.
print(f"idx is: {idx}")
print(f"eigenvalues is:\n {eigenvalues}")
print(f"eigenvectors is:\n {eigenvectors}")

M_00_inv is: [[ 1.05777482 -0.73427448  0.04302633]
 [-0.73427448  1.54049244 -0.08088591]
 [ 0.04302633 -0.08088591  0.96884372]]
A is: [[ 0.37539851  0.1863545  -0.04933069]
 [ 0.1863545   0.27192556  0.01256588]
 [-0.04933069  0.01256588  0.44736033]]
B is: [[79.10558135 63.62950556 -5.5509043 ]
 [63.62950556 51.60085275 -4.41781241]
 [-5.5509043  -4.41781241 15.74730922]]
eigenvalues, eigenvectors are:
 [0.51321233+0.j 0.00395371+0.j 0.02821164+0.j],
 [[-0.62803251 -0.72063358  0.22865848]
 [ 0.77818428 -0.69331036 -0.20032219]
 [ 0.00209368  0.00282546  0.95267325]]
idx is: [0 2 1]
eigenvalues is:
 [0.51321233 0.02821164 0.00395371]
eigenvectors is:
 [[-0.62803251  0.22865848 -0.72063358]
 [ 0.77818428 -0.20032219 -0.69331036]
 [ 0.00209368  0.95267325  0.00282546]]


In [65]:
print(f"--- Step 1: Johansen Sorted Eigenvalues ---")
for i, lam in enumerate(eigenvalues):
    print(f"Eigenvalue \\lambda_{i+1}: {lam:.4f}")

--- Step 1: Johansen Sorted Eigenvalues ---
Eigenvalue \lambda_1: 0.5132
Eigenvalue \lambda_2: 0.0282
Eigenvalue \lambda_3: 0.0040


In [73]:
# 5. Extract the Cointegrating Vector (Beta)
# the first eigenvector corresponding to the largest eigenvalue forms our main equilibrium vector
beta_1 = eigenvectors[:,0]
print(beta_1)

# Normalize the vector relative to the first variable for easy reading.
beta_1_normalized = beta_1 / beta_1[0]
print(beta_1_normalized)

print(f"\n--- Step 2: Cointegrating Vector (\\u03b2_1)---")
print(f"Normalized \\u03b2 vector: {beta_1_normalized}")
print(f"Long-run stationary relation: ({beta_1_normalized[0]:.2f} * z1) + ({beta_1_normalized[1]:.2f} * z2) + ({beta_1_normalized[2]:.2f} * z3) = Stationary Equilibrium")

# 6. Calculate Speed of adjustment (\u03b1)
# \u03b1 = M_01 * \u03b2 * (\u03b2' * M_11 * \u03b2)^-1
alpha_1 = M_01 @ beta_1 / (beta_1.T @ M_11 @ beta_1)
print("\n-- Step 3: Speed of Adjustment (\\uo3b1) ---")
for i, alp in enumerate(alpha_1):
    print(f"Economy variable z{i+1} correction speed: {alp:.4f}")

[-0.62803251  0.77818428  0.00209368]
[ 1.         -1.2390828  -0.00333372]

--- Step 2: Cointegrating Vector (\u03b2_1)---
Normalized \u03b2 vector: [ 1.         -1.2390828  -0.00333372]
Long-run stationary relation: (1.00 * z1) + (-1.24 * z2) + (-0.00 * z3) = Stationary Equilibrium

-- Step 3: Speed of Adjustment (\uo3b1) ---
Economy variable z1 correction speed: 0.4340
Economy variable z2 correction speed: -0.8885
Economy variable z3 correction speed: 0.1128
